In [1]:
class PlayerProfile:
    def __init__(self, name, team, position, overall_rating):
        self.name = name
        self.team = team
        self.position = position.lower()
        self.overall_rating = overall_rating
        self.base_rating = 6
        self.match_rating = self.base_rating

    def adjust_rating(self, delta):
        self.match_rating = max(0, min(10, self.match_rating + delta))

    def get_rating_boost(self):
        if self.overall_rating >= 87:
            return 0.2
        elif self.overall_rating >= 84:
            return 0.25
        elif self.overall_rating >= 81:
            return 0.3
        elif self.overall_rating >= 78:
            return 0.35
        elif self.overall_rating >= 75:
            return 0.40
        else:
            return 0.45

    def is_forward(self): return self.position == "forward"
    def is_midfielder(self): return self.position == "midfielder"
    def is_defender(self): return self.position == "defender"

    def input_match_stats(self, minutes=0, goals=0, assists=0,
                          yellow_card=False, red_card=False, clean_sheet=False, goals_conceded=0,
                          total_shots=0, shots_on_target=0, shots_off_target=0, tackles_loss=0,
                          total_passes=0, accurate_passes=0,
                          expected_goals=0, expected_assists=0,
                          big_chances_missed=0,
                          successful_dribbles=0, total_dribbles=0, conceded_penalty=0, missed_penalty=0,
                          accurate_crosses=0, total_crosses=0,
                          accurate_long_balls=0, total_long_balls=0,
                          dispossessed=0, tackles_won=0, interceptions=0,
                          clearances=0, ball_recoveries=0,
                          dribbled_past=0, duels_won=0, duels_lost=0,
                          ground_duels_won=0, ground_duels_total=0,
                          aerial_duels_won=0, aerial_duels_total=0, own_goal=0,
                          fouled=0, number_of_fouls=0):

        boost = self.get_rating_boost()

        # Goals and Assists
        self.adjust_rating(goals * 1)
        self.adjust_rating(assists * 1)

        if goals_conceded > 2:
            self.adjust_rating(-1)
            

        
        # Cards
        if yellow_card:
            self.adjust_rating(-1)
        if red_card:
            self.adjust_rating(-2)
                #Clean Sheet Bonus
        if clean_sheet:
            self.adjust_rating(boost)

        # Shot Accuracy
        if total_shots > 0:
            accuracy = shots_on_target / total_shots
            if accuracy >= 0.7:
                self.adjust_rating(boost)
            shots_off_target = total_shots - shots_on_target
            off_target_ratio = shots_off_target / total_shots
            if off_target_ratio >= 0.75:
                self.adjust_rating(-0.3)
            elif off_target_ratio >= 0.5:
                self.adjust_rating(-0.2)
            elif off_target_ratio >= 0.3:
                self.adjust_rating(-0.1)

        # xG comparison
        if goals > expected_goals:
            self.adjust_rating(boost)
        elif expected_goals > goals:
            self.adjust_rating(-0.2)

        # xA comparison
        if assists > expected_assists:
            self.adjust_rating(boost)
        elif expected_assists > assists:
            self.adjust_rating(-0.2)

        # Passing
        if total_passes > 0:
            pass_accuracy = accurate_passes / total_passes
            if pass_accuracy >= 0.9:
                self.adjust_rating(boost)
            elif pass_accuracy >= 0.75:
                self.adjust_rating(boost * 0.75)
            elif pass_accuracy >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.3)

        # Dribbling Success Rate
        if total_dribbles > 0:
            dribble_rate = successful_dribbles / total_dribbles
            if dribble_rate >= 0.8:
                self.adjust_rating(boost)
            elif dribble_rate >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.2)
            if tackles_won > tackles_loss:
                self.adjust_rating(boost)
            elif tackles_won < tackles_loss:
                self.adjust_rating(-boost)
             
        # Crossing
        if total_crosses > 0:
            crossing_rate = accurate_crosses / total_crosses
            if crossing_rate >= 0.5:
                self.adjust_rating(boost)
            elif crossing_rate >= 0.3:
                self.adjust_rating(boost * 0.5)

        # Long Balls
        if total_long_balls > 0:
            long_ball_rate = accurate_long_balls / total_long_balls
            if long_ball_rate >= 0.5:
                self.adjust_rating(boost)
            elif long_ball_rate >= 0.3:
                self.adjust_rating(boost * 0.5)

        # Possession Loss
        if dispossessed >= 3:
            self.adjust_rating(-0.1 * dispossessed)

        # Defensive Metrics
        if tackles_won >= 2:
            self.adjust_rating(tackles_won * 0.1)
        if interceptions >= 2:
            self.adjust_rating(interceptions * 0.1)
        if clearances >= 2:
            self.adjust_rating(clearances * 0.05)
        if ball_recoveries >= 5:
            self.adjust_rating(ball_recoveries * 0.05)
        if dribbled_past >= 2:
            self.adjust_rating(-0.1 * dribbled_past)

        # Duels
        total_duels = duels_won + duels_lost
        if total_duels > 0:
            duel_win_rate = duels_won / total_duels
            if duel_win_rate >= 0.6:
                self.adjust_rating(boost)
            elif duel_win_rate < 0.4:
                self.adjust_rating(-boost)

        if ground_duels_total > 0:
            ground_rate = ground_duels_won / ground_duels_total
            if ground_rate >= 0.6:
                self.adjust_rating(boost * 0.5)

        if aerial_duels_total > 0:
            aerial_rate = aerial_duels_won / aerial_duels_total
            if aerial_rate >= 0.6:
                self.adjust_rating(boost * 0.5)

        # Fouls and Being Fouled
        self.adjust_rating(fouled * 0.05)
        self.adjust_rating(-number_of_fouls * 0.1)

        # Played Full 90 Minutes
        if minutes >= 90:
            self.adjust_rating(0.2)

        # Big Chances Missed
        if big_chances_missed > 0:
            self.adjust_rating(-0.3 * big_chances_missed)

        if conceded_penalty > 0:
            self.adjust_rating(-2)
        if missed_penalty > 0:
            self.adjust_rating(-2)
        if minutes >= 90:
            self.adjust_rating(0.2) 
        if own_goal > 0:
            self.adjust_rating(-2)

class GoalkeeperProfile:
    def __init__(self, name, team, position, overall_rating):
        self.name = name
        self.team = team
        self.position = position.lower()
        self.overall_rating = overall_rating
        self.base_rating = 6
        self.match_rating = self.base_rating

    def adjust_rating(self, delta):
        self.match_rating = max(0, min(10, self.match_rating + delta))

    def get_rating_boost(self):
        if self.overall_rating >= 87:
            return 0.2
        elif self.overall_rating >= 84:
            return 0.25
        elif self.overall_rating >= 81:
            return 0.3
        elif self.overall_rating >= 78:
            return 0.35
        elif self.overall_rating >= 75:
            return 0.4
        else:
            return 0.5

    def input_match_stats(self,
                          saves=0,
                          goals_conceded=0,
                          xG_faced=0,
                          saves_inside_box=0,
                          saves_outside_box=0,
                          touches=0,
                          goals_prevented=0,
                          errors=0,
                          minutes_played=0,
                          penalty_saves=0,
                          total_passes=0,
                          accurate_passes=0,
                          total_long_balls=0,
                          accurate_long_balls=0,
                          yellow_card=False,
                          red_card=False,
                          clean_sheet=False):

        boost = self.get_rating_boost()

        # Saves
        self.adjust_rating(saves * 0.1)
        if saves_inside_box > 0:
            self.adjust_rating(saves_inside_box * 0.15)
        if saves_outside_box > 0:
            self.adjust_rating(saves_outside_box * 0.1)

        # Goals Conceded vs xG
        if goals_conceded < xG_faced:
            self.adjust_rating(boost)
        elif goals_conceded > xG_faced:
            self.adjust_rating(-boost)

                # Passing
        if total_passes > 0:
            pass_accuracy = accurate_passes / total_passes
            if pass_accuracy >= 0.9:
                self.adjust_rating(boost)
            elif pass_accuracy >= 0.75:
                self.adjust_rating(boost * 0.75)
            elif pass_accuracy >= 0.5:
                self.adjust_rating(boost * 0.5)
            else:
                self.adjust_rating(-0.3)

        # Goals prevented
        if goals_prevented > 0:
            self.adjust_rating(goals_prevented * 0.2)

        # Errors
        if errors == 0:
            self.adjust_rating(boost * 0.5)
        elif errors > 0:
            self.adjust_rating(-0.5 * errors)

        # Touches which can lead to distribution
        if touches >= 40:
            self.adjust_rating(boost * 0.5)
        elif touches >= 25:
            self.adjust_rating(boost * 0.2)

        # Penalty saves
        if penalty_saves > 0:
            self.adjust_rating(penalty_saves * 0.8)

        # Clean sheet 
        if clean_sheet and minutes_played >= 70:
            self.adjust_rating(0.5)

        # Played full match
        if minutes_played >= 90:
            self.adjust_rating(0.2)

        
        # Cards
        if yellow_card:
            self.adjust_rating(-1)
        if red_card:
            self.adjust_rating(-2)


In [3]:
#Canada
canada_world_cup_starting_xi_vs_bosnia= [
    ["Maxime Crepeau", "GK", "Orlando City", 70],
    ["Richie Laryea", "LB", "Toronto", 71,],
    ["Derek Cornelius", "CB", "Marseille", 74,],
    ["Luc de Fougerolles", "CB", "Fulham", 64,],
    ["Alstair Johnston", "RB", "Celtic", 78,],
    ["Tajon Buchanan", "RM", "Villareal", 77,],
    ["Ismael Kone", "CM", "Sassuolo", 78,],
    ["Stephen Eustaquio", "CM", "Porto", 76,],
    ["Liam Millar", "LM", "Hull City", 73,],
    ["Jonathan David", "ST", "Juventus", 80,],
    ["Tani Oluwaseyi", "ST", "Villarreal", 72,],
    
] 
canada_world_cup_bench_vs_bosnia= [
    [" Ali Ahmed", "LM", "Norwich", 70,],
    ["Promise David", "ST", "Union SG", 75,],
    ["Cyle Larin", "ST", "Southampton", 76,],
    ["Jacob Shaffelburg", "RM", "Toronto", 66,],
    ["Jonathan Osorio", "CM", "Toronto", 71,],
]


In [15]:
gk = GoalkeeperProfile("Crepeau", "Canada", "goalkeeper", 70)
gk.input_match_stats(
    minutes_played=90,
    saves=2,
    saves_inside_box=2,
    saves_outside_box=0,
    goals_conceded=1,
    xG_faced=1,
    goals_prevented=0,
    total_passes=24,
    accurate_passes=17,
    total_long_balls=9,
    accurate_long_balls=2,
    touches=31,
    errors=0,
    penalty_saves=0,
    clean_sheet=False,
    yellow_card=False
)
print(f"{gk.name}'s match rating: {gk.match_rating:.2f}")

Crepeau's match rating: 7.30


In [17]:
player = PlayerProfile("Laryea", "Canada", "defender", 71)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=27,
    total_passes=37,
    expected_goals=0.26,
    expected_assists=0.43,
    successful_dribbles=2,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=2,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=5,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=4,
    dribbled_past=0,
    duels_won=9,
    duels_lost=2,
    ground_duels_won=8,
    ground_duels_total=8,
    aerial_duels_won=1,
    aerial_duels_total=3,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Laryea's match rating: 9.25


In [19]:
player = PlayerProfile("Luc de Fougerolles", "Canada", "defender", 64)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=39,
    total_passes=50,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=3,
    total_long_balls=6,
    dispossessed=0,
    tackles_won=3,
    tackles_loss=0,
    clearances=8,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=11,
    duels_lost=10,
    ground_duels_won=7,
    ground_duels_total=9,
    aerial_duels_won=4,
    aerial_duels_total=13,
    fouled=4,
    number_of_fouls=2,
    yellow_card=True,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Luc de Fougerolles's match rating: 6.91


In [21]:
player = PlayerProfile("Derek Cornelius", "Canada", "defender", 74)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=50,
    total_passes=66,
    expected_goals=0.01,
    expected_assists=0.06,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=6,
    dispossessed=0,
    tackles_won=4,
    tackles_loss=0,
    clearances=4,
    interceptions=1,
    ball_recoveries=7,
    dribbled_past=0,
    duels_won=9,
    duels_lost=7,
    ground_duels_won=5,
    ground_duels_total=5,
    aerial_duels_won=4,
    aerial_duels_total=11,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Derek Cornelius's match rating: 7.26


In [23]:
player = PlayerProfile("Johnston", "Canada", "defender", 78)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=24,
    total_passes=36,
    expected_goals=0.06,
    expected_assists=0.05,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=4,
    total_long_balls=7,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=4,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=5,
    duels_lost=2,
    ground_duels_won=4,
    ground_duels_total=6,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=2,
    number_of_fouls=2,
    yellow_card=True,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Johnston's match rating: 6.72


In [27]:
player = PlayerProfile("Eustaquio", "Canada", "midfielder", 76)

player.input_match_stats(
    minutes=89,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=42,
    total_passes=47,
    expected_goals=0.01,
    expected_assists=0.41,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=4,
    total_crosses=9,
    accurate_long_balls=2,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=9,
    dribbled_past=0,
    duels_won=4,
    duels_lost=3,
    ground_duels_won=4,
    ground_duels_total=6,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=3,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Eustaquio's match rating: 6.80


In [29]:
player = PlayerProfile("Kone", "Canada", "midfielder", 78)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=50,
    total_passes=59,
    expected_goals=0.02,
    expected_assists=0.06,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=3,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=2,
    ball_recoveries=5,
    dribbled_past=3,
    duels_won=4,
    duels_lost=13,
    ground_duels_won=3,
    ground_duels_total=11,
    aerial_duels_won=1,
    aerial_duels_total=6,
    fouled=2,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Kone's match rating: 6.26


In [33]:
player = PlayerProfile("Buchanan", "Canada", "midfielder", 77)

player.input_match_stats(
    minutes=61,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=6,
    total_passes=7,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=3,
    accurate_crosses=0,
    total_crosses=3,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=2,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=3,
    duels_lost=7,
    ground_duels_won=3,
    ground_duels_total=8,
    aerial_duels_won=0,
    aerial_duels_total=2,
    fouled=2,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Buchanan's match rating: 6.00


In [35]:
player = PlayerProfile("Millar", "Canada", "midfielder", 73)

player.input_match_stats(
    minutes=61,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=21,
    total_passes=28,
    expected_goals=0,
    expected_assists=0.08,
    successful_dribbles=2,
    total_dribbles=3,
    accurate_crosses=0,
    total_crosses=4,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=2,
    duels_won=6,
    duels_lost=7,
    ground_duels_won=4,
    ground_duels_total=8,
    aerial_duels_won=2,
    aerial_duels_total=5,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Millar's match rating: 6.71


In [37]:
player = PlayerProfile("Jonathan David", "Canada", "forward", 80)

player.input_match_stats(
    minutes=61,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=1,
    accurate_passes=8,
    total_passes=14,
    expected_goals=0.39,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=1,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=2,
    duels_lost=2,
    ground_duels_won=2,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Jonathan David's match rating: 6.15


In [39]:
player = PlayerProfile("Oluwaseyi", "Canada", "forward", 72)

player.input_match_stats(
    minutes=76,
    goals=0,
    assists=0,
    total_shots=2,
    shots_on_target=1,
    accurate_passes=6,
    total_passes=20,
    expected_goals=0.18,
    expected_assists=0,
    successful_dribbles=1,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=8,
    duels_lost=11,
    ground_duels_won=3,
    ground_duels_total=6,
    aerial_duels_won=5,
    aerial_duels_total=13,
    fouled=2,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Oluwaseyi's match rating: 6.07


In [41]:
player = PlayerProfile("Alli", "Canada", "midfielder", 70)

player.input_match_stats(
    minutes=29,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=10,
    total_passes=18,
    expected_goals=0,
    expected_assists=0.02,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=1,
    total_long_balls=2,
    dispossessed=1,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=1,
    duels_won=2,
    duels_lost=3,
    ground_duels_won=2,
    ground_duels_total=4,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Alli's match rating: 7.38


In [43]:
player = PlayerProfile("Promise David", "Canada", "forward", 75)

player.input_match_stats(
    minutes=29,
    goals=0,
    assists=1,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=1,
    total_passes=3,
    expected_goals=0.04,
    expected_assists=0.05,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=3,
    tackles_won=1,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=3,
    duels_lost=7,
    ground_duels_won=2,
    ground_duels_total=7,
    aerial_duels_won=1,
    aerial_duels_total=3,
    fouled=1,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Promise David's match rating: 5.75


In [45]:
player = PlayerProfile("Larin", "Canada", "forward", 76)

player.input_match_stats(
    minutes=14,
    goals=1,
    assists=0,
    total_shots=1,
    shots_on_target=1,
    accurate_passes=0,
    total_passes=0,
    expected_goals=0.21,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=0,
    ball_recoveries=1,
    dribbled_past=0,
    duels_won=2,
    duels_lost=1,
    ground_duels_won=1,
    ground_duels_total=1,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Larin's match rating: 8.45


In [47]:
player = PlayerProfile("Shaffelburg", "Canada", "midfielder", 68)

player.input_match_stats(
    minutes=29,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=5,
    total_passes=7,
    expected_goals=0,
    expected_assists=0.14,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=1,
    total_crosses=1,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=2,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=1,
    duels_lost=3,
    ground_duels_won=1,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Shaffelburg's match rating: 6.02


In [49]:
player = PlayerProfile("Osorio", "Canada", "midfielder", 71)

player.input_match_stats(
    minutes=1,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=4,
    total_passes=5,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=0,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Osorio's match rating: 6.14


In [ ]:
#Bosnia
bosnia_world_cup_starting_xi_vs_canada = [
    ["Nikola Vasilj", "GK", "St Pauli", 78,],
    ["Amar Dedic", "RB", "Benfica", 77,],
    ["Tarik Muharemovic ", "CB", "Sassuolo", 77,],
    ["Nikola Katic", "CB", "Schalke 04", 71,],
    ["Sead Kolasinac", "LB", "Atalanta", 78,],
    ["Esmir Bajraktarevic", "RW", "PSV Eindhoven", 73,],
    ["Ivan Basic", "CM", "Astana", 70 ,],
    ["Benjamin Tahirovic", "CM", "Brondby", 70,],
    ["Amar Memic", "LM", "Viktoria Plzen", 74,],
    ["Ermedin Demirovic", "ST", "VFB Stuggart", 80,],
    ["Jovo Lukic", "ST", "Universitatea Cluj", 72,],

]

bosnia_world_cup_bench_vs_canada = [
    ["Samed Bazdar", "ST", "Jagiellonia Bialystok", 69, , None],
    ["Kerim Alajbegovic", "LW", "RB Salzburg", 74, , None],
    ["Dzenis Burnic", "CM", "Karlsruher SC", 71, , None]
    ["Ivan Sunjic", "CDM", "Pafos", 73, , None],
    ["Armin Gigovic", "CM", "Young Boys", 72, , None],
]   

In [53]:
gk = GoalkeeperProfile("Vasilj", "Bosnia", "goalkeeper", 78)
gk.input_match_stats(
    minutes_played=90,
    saves=1,
    saves_inside_box=1,
    saves_outside_box=0,
    goals_conceded=1,
    xG_faced=0.75,
    goals_prevented=-0.25,
    total_passes=30,
    accurate_passes=11,
    total_long_balls=24,
    accurate_long_balls=5,
    touches=38,
    errors=0,
    penalty_saves=0,
    clean_sheet=False,
    yellow_card=False
)
print(f"{gk.name}'s match rating: {gk.match_rating:.2f}")

Vasilj's match rating: 6.05


In [61]:
player = PlayerProfile("Dedic", "Bosnia", "defender", 77)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=14,
    total_passes=22,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=4,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=7,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=1,
    duels_won=6,
    duels_lost=4,
    ground_duels_won=3,
    ground_duels_total=7,
    aerial_duels_won=3,
    aerial_duels_total=3,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Dedic's match rating: 7.70


In [59]:
player = PlayerProfile("Katic", "Bosnia", "defender", 71)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=14,
    total_passes=23,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=2,
    total_long_balls=7,
    dispossessed=0,
    tackles_won=5,
    tackles_loss=0,
    clearances=16,
    interceptions=3,
    ball_recoveries=2,
    dribbled_past=2,
    duels_won=15,
    duels_lost=9,
    ground_duels_won=5,
    ground_duels_total=9,
    aerial_duels_won=10,
    aerial_duels_total=15,
    fouled=0,
    number_of_fouls=2,
    yellow_card=True,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Katic's match rating: 7.50


In [57]:
player = PlayerProfile("Muharemovic", "Bosnia", "defender", 77)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=24,
    total_passes=31,
    expected_goals=0.18,
    expected_assists=0,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=2,
    total_long_balls=6,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=21,
    interceptions=2,
    ball_recoveries=6,
    dribbled_past=0,
    duels_won=9,
    duels_lost=2,
    ground_duels_won=3,
    ground_duels_total=4,
    aerial_duels_won=6,
    aerial_duels_total=7,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Muharemovic's match rating: 9.50


In [55]:
player = PlayerProfile("Kolasinac", "Bosnia", "defender", 78)

player.input_match_stats(
    minutes=83,
    goals=0,
    assists=1,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=0,
    total_passes=0,
    expected_goals=0,
    expected_assists=0.51,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=1,
    total_crosses=2,
    accurate_long_balls=1,
    total_long_balls=2,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=7,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=5,
    duels_lost=4,
    ground_duels_won=4,
    ground_duels_total=7,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=2,
    number_of_fouls=3,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=1 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Kolasinac's match rating: 8.40


In [63]:
player = PlayerProfile("Bajraktarevic", "Bosnia", "midfielder", 73)

player.input_match_stats(
    minutes=74,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=7,
    total_passes=15,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=1,
    total_dribbles=5,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=1,
    tackles_won=0,
    tackles_loss=0,
    clearances=3,
    interceptions=2,
    ball_recoveries=4,
    dribbled_past=2,
    duels_won=3,
    duels_lost=10,
    ground_duels_won=2,
    ground_duels_total=10,
    aerial_duels_won=1,
    aerial_duels_total=3,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Bajraktarevic's match rating: 5.15


In [65]:
player = PlayerProfile("Basic", "Bosnia", "midfielder", 70)

player.input_match_stats(
    minutes=62,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=17,
    total_passes=21,
    expected_goals=0,
    expected_assists=0.29,
    successful_dribbles=1,
    total_dribbles=1,
    accurate_crosses=5,
    total_crosses=5,
    accurate_long_balls=2,
    total_long_balls=3,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=1,
    interceptions=2,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=2,
    duels_lost=3,
    ground_duels_won=1,
    ground_duels_total=3,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=0,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Basic's match rating: 7.49


In [67]:
player = PlayerProfile("Tahirovic", "Bosnia", "midfielder", 70)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=17,
    total_passes=24,
    expected_goals=0.02,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=2,
    accurate_long_balls=1,
    total_long_balls=6,
    dispossessed=0,
    tackles_won=3,
    tackles_loss=0,
    clearances=3,
    interceptions=1,
    ball_recoveries=5,
    dribbled_past=1,
    duels_won=5,
    duels_lost=3,
    ground_duels_won=4,
    ground_duels_total=6,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Tahirovic's match rating: 7.35


In [69]:
player = PlayerProfile("Memic", "Bosnia", "midfielder", 74)

player.input_match_stats(
    minutes=74,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=3,
    total_passes=7,
    expected_goals=0.05,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=2,
    dispossessed=1,
    tackles_won=4,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=1,
    duels_won=2,
    duels_lost=6,
    ground_duels_won=1,
    ground_duels_total=6,
    aerial_duels_won=1,
    aerial_duels_total=2,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Memic's match rating: 5.35


In [71]:
player = PlayerProfile("Demirovic", "Bosnia", "forward", 80)

player.input_match_stats(
    minutes=90,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=12,
    total_passes=17,
    expected_goals=0.02,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=1,
    tackles_won=3,
    tackles_loss=0,
    clearances=3,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=11,
    duels_lost=11,
    ground_duels_won=4,
    ground_duels_total=11,
    aerial_duels_won=7,
    aerial_duels_total=11,
    fouled=1,
    number_of_fouls=4,
    yellow_card=True,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Demirovic's match rating: 5.50


In [73]:
player = PlayerProfile("Lukic", "Bosnia", "forard", 72)

player.input_match_stats(
    minutes=62,
    goals=1,
    assists=0,
    total_shots=3,
    shots_on_target=2,
    accurate_passes=7,
    total_passes=17,
    expected_goals=0.99,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=1,
    tackles_loss=0,
    clearances=3,
    interceptions=0,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=10,
    duels_lost=3,
    ground_duels_won=1,
    ground_duels_total=4,
    aerial_duels_won=9,
    aerial_duels_total=9,
    fouled=0,
    number_of_fouls=1,
    yellow_card=True,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Lukic's match rating: 7.03


In [75]:
player = PlayerProfile("Gigovic", "Bosnia", "midfielder", 72)

player.input_match_stats(
    minutes=28,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=12,
    total_passes=16,
    expected_goals=0,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=0,
    tackles_won=2,
    tackles_loss=0,
    clearances=1,
    interceptions=1,
    ball_recoveries=3,
    dribbled_past=0,
    duels_won=5,
    duels_lost=3,
    ground_duels_won=4,
    ground_duels_total=7,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=1,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Gigovic's match rating: 7.06


In [77]:
player = PlayerProfile("Bazdar", "Bosnia", "forward", 69)

player.input_match_stats(
    minutes=28,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=2,
    total_passes=6,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=1,
    total_dribbles=3,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=0,
    dispossessed=1,
    tackles_won=1,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=4,
    duels_lost=9,
    ground_duels_won=3,
    ground_duels_total=7,
    aerial_duels_won=1,
    aerial_duels_total=6,
    fouled=1,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Bazdar's match rating: 5.45


In [79]:
player = PlayerProfile("Sunjic", "Bosnia", "midfielder", 73)

player.input_match_stats(
    minutes=16,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=10,
    total_passes=14,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=0,
    total_long_balls=2,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=1,
    duels_lost=2,
    ground_duels_won=0,
    ground_duels_total=2,
    aerial_duels_won=1,
    aerial_duels_total=1,
    fouled=0,
    number_of_fouls=2,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Sunjic's match rating: 5.80


In [81]:
player = PlayerProfile("Alajbegovic", "Bosnia", "midfielder", 74)

player.input_match_stats(
    minutes=16,
    goals=0,
    assists=0,
    total_shots=0,
    shots_on_target=0,
    accurate_passes=5,
    total_passes=5,
    expected_goals=0,
    expected_assists=0,
    successful_dribbles=1,
    total_dribbles=2,
    accurate_crosses=0,
    total_crosses=1,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=2,
    dribbled_past=0,
    duels_won=1,
    duels_lost=2,
    ground_duels_won=1,
    ground_duels_total=3,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=1,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Alajbegovic's match rating: 6.58


In [83]:
player = PlayerProfile("Burnic", "Bosnia", "midfielder", 71)

player.input_match_stats(
    minutes=6,
    goals=0,
    assists=0,
    total_shots=1,
    shots_on_target=0,
    accurate_passes=2,
    total_passes=2,
    expected_goals=0.01,
    expected_assists=0.01,
    successful_dribbles=0,
    total_dribbles=0,
    accurate_crosses=0,
    total_crosses=0,
    accurate_long_balls=1,
    total_long_balls=1,
    dispossessed=0,
    tackles_won=0,
    tackles_loss=0,
    clearances=0,
    interceptions=0,
    ball_recoveries=0,
    dribbled_past=0,
    duels_won=0,
    duels_lost=0,
    ground_duels_won=0,
    ground_duels_total=0,
    aerial_duels_won=0,
    aerial_duels_total=0,
    fouled=0,
    number_of_fouls=0,
    yellow_card=False,
    red_card=False,
    clean_sheet=False, #Just for Defenders
    goals_conceded=0 #Just for Defenders
               
)

print(f"{player.name}'s match rating: {player.match_rating:.2f}")

Burnic's match rating: 6.20
